In [1]:
import pandas as pd

In [2]:
with open("access.log" ,"r") as file:
    lines = file.readlines()

logs_data=pd.DataFrame(lines,columns=['log'])

In [3]:
logs_data

,log
0,in24.inetnebr.com - - [01/Aug/1995:00:00:01 -0...
1,uplherc.upl.com - - [01/Aug/1995:00:00:07 -040...
2,uplherc.upl.com - - [01/Aug/1995:00:00:08 -040...
3,uplherc.upl.com - - [01/Aug/1995:00:00:08 -040...
4,uplherc.upl.com - - [01/Aug/1995:00:00:08 -040...
...,...
1569893,gatekeeper.uccu.com - - [31/Aug/1995:23:59:49 ...
1569894,gatekeeper.uccu.com - - [31/Aug/1995:23:59:49 ...
1569895,cys-cap-9.wyoming.com - - [31/Aug/1995:23:59:5...
1569896,www-c8.proxy.aol.com - - [31/Aug/1995:23:59:52...


In [4]:
pattern = r'(\S+) \S+ \S+ \[(.*?)\] "(\S+) (.*?) (\S+)" (\d{3}) (\S+)'

logs_data[['ip_address', 'time_stamp', 'http_method_type', 'requested_url', 'protocol', 'status', 'size']] = (
    logs_data['log'].str.extract(pattern)
)

In [5]:
logs_data.head()

,log,ip_address,time_stamp,http_method_type,requested_url,protocol,status,size
0,in24.inetnebr.com - - [01/Aug/1995:00:00:01 -0...,in24.inetnebr.com,01/Aug/1995:00:00:01 -0400,GET,/shuttle/missions/sts-68/news/sts-68-mcc-05.txt,HTTP/1.0,200,1839
1,uplherc.upl.com - - [01/Aug/1995:00:00:07 -040...,uplherc.upl.com,01/Aug/1995:00:00:07 -0400,GET,/,HTTP/1.0,304,0
2,uplherc.upl.com - - [01/Aug/1995:00:00:08 -040...,uplherc.upl.com,01/Aug/1995:00:00:08 -0400,GET,/images/ksclogo-medium.gif,HTTP/1.0,304,0
3,uplherc.upl.com - - [01/Aug/1995:00:00:08 -040...,uplherc.upl.com,01/Aug/1995:00:00:08 -0400,GET,/images/MOSAIC-logosmall.gif,HTTP/1.0,304,0
4,uplherc.upl.com - - [01/Aug/1995:00:00:08 -040...,uplherc.upl.com,01/Aug/1995:00:00:08 -0400,GET,/images/USA-logosmall.gif,HTTP/1.0,304,0


# Data Cleaning

### IP ADDRESS COLUMN

In [6]:
logs_data["ip_address"].unique()

array(['in24.inetnebr.com', 'uplherc.upl.com',
       'ix-esc-ca2-07.ix.netcom.com', ..., 'dffl6-14.gate.net',
       '137.229.105.152', '198.146.93.224'], dtype=object)

In [7]:
logs_data["ip_address"].value_counts()

ip_address
edams.ksc.nasa.gov         6528
piweba4y.prodigy.com       4846
163.206.89.4               4791
piweba5y.prodigy.com       4607
piweba3y.prodigy.com       4416
                           ... 
199.172.142.107               1
133.56.112.12                 1
sojus.strm.ing.tu-bs.de       1
ccmac1.unican.es              1
mcimcwk12.san.uc.edu          1
Name: count, Length: 74969, dtype: int64

### HTTP METHOD COLUMN

In [8]:
logs_data["http_method_type"].unique()

array(['GET', nan, 'POST', 'HEAD'], dtype=object)

In [9]:
logs_data["http_method_type"].value_counts()

http_method_type
GET     1563829
HEAD       3965
POST        111
Name: count, dtype: int64

### TIME STAMP COLUMN 

In [10]:
logs_data["time_stamp"].unique()

array(['01/Aug/1995:00:00:01 -0400', '01/Aug/1995:00:00:07 -0400',
       '01/Aug/1995:00:00:08 -0400', ..., '31/Aug/1995:23:59:49 -0400',
       '31/Aug/1995:23:59:52 -0400', '31/Aug/1995:23:59:53 -0400'],
      dtype=object)

In [11]:
logs_data["requested_url"].unique()

array(['/shuttle/missions/sts-68/news/sts-68-mcc-05.txt', '/',
       '/images/ksclogo-medium.gif', ...,
       '/cgi-bin/imagemap/index69?675,97',
       '/cgi-bin/imagemap/index69?682,101',
       '/cgi-bin/imagemap/index69?682,144'], dtype=object)

### PROTOCOL COLUMN

In [12]:
logs_data["protocol"].value_counts()

protocol
HTTP/1.0                              1567112
<berend@blazemonger.pc.cc.cmu.edu>        624
HTTP/V1.0                                 163
(MILA)</a>                                  1
headers                                     1
of                                          1
Ships                                       1
a                                           1
homepage</a><hr>                            1
Name: count, dtype: int64

In [13]:
valid_protocols = ["HTTP/1.0", "HTTP/1.1"]

invalid = logs_data[~logs_data["protocol"].isin(valid_protocols)]

print(invalid[["log", "protocol"]].count())

log         2786
protocol     793
dtype: int64


In [14]:
logs_data = logs_data[
    logs_data["protocol"].isin(["HTTP/1.0", "HTTP/1.1"])
]

In [15]:
logs_data["protocol"].value_counts()

protocol
HTTP/1.0    1567112
Name: count, dtype: int64

### STATUS COLUMN 

In [16]:
logs_data["status"].unique()

array(['200', '304', '302', '404', '403', '500', '501'], dtype=object)

In [17]:
logs_data["status"].value_counts()

status
200    1396349
304     134146
302      26472
404       9944
403        171
501         27
500          3
Name: count, dtype: int64

In [18]:
logs_data["size"].value_counts()

size
0         140410
786        77706
1204       63689
363        58289
234        57952
           ...  
36716          1
142983         1
1649           1
1658           1
1663           1
Name: count, Length: 7623, dtype: int64

In [37]:
logs_data = logs_data[logs_data["size"].str.contains(r'\d') != False]

In [19]:
logs_data = logs_data.drop(columns ="log")

In [38]:
logs_data.head()

,ip_address,time_stamp,http_method_type,requested_url,protocol,status,size
0,in24.inetnebr.com,1995-08-01 00:00:01-04:00,GET,/shuttle/missions/sts-68/news/sts-68-mcc-05.txt,HTTP/1.0,200,1839
1,uplherc.upl.com,1995-08-01 00:00:07-04:00,GET,/,HTTP/1.0,304,0
2,uplherc.upl.com,1995-08-01 00:00:08-04:00,GET,/images/ksclogo-medium.gif,HTTP/1.0,304,0
3,uplherc.upl.com,1995-08-01 00:00:08-04:00,GET,/images/MOSAIC-logosmall.gif,HTTP/1.0,304,0
4,uplherc.upl.com,1995-08-01 00:00:08-04:00,GET,/images/USA-logosmall.gif,HTTP/1.0,304,0


# DATA TYPE CORRECTION

In [40]:
logs_data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1553068 entries, 0 to 1569897
Data columns (total 7 columns):
 #   Column            Non-Null Count    Dtype                    
---  ------            --------------    -----                    
 0   ip_address        1553068 non-null  object                   
 1   time_stamp        1553068 non-null  datetime64[ns, UTC-04:00]
 2   http_method_type  1553068 non-null  object                   
 3   requested_url     1553068 non-null  object                   
 4   protocol          1553068 non-null  object                   
 5   status            1553068 non-null  int64                    
 6   size              1553068 non-null  int64                    
dtypes: datetime64[ns, UTC-04:00](1), int64(2), object(4)
memory usage: 94.8+ MB


In [22]:
logs_data["time_stamp"] = pd.to_datetime(logs_data["time_stamp"], format="%d/%b/%Y:%H:%M:%S %z")
logs_data["status"] = logs_data["status"].astype(int)
logs_data["size"] =logs_data["size"].astype(int)

### CHECK AND REMOVE NULL VALUES 

In [41]:
logs_data.isnull().sum()

ip_address          0
time_stamp          0
http_method_type    0
requested_url       0
protocol            0
status              0
size                0
dtype: int64

### CHECK AND REMOVE DUPLICATE ROWS

In [47]:
logs_data[logs_data.duplicated(keep=False)]

,ip_address,time_stamp,http_method_type,requested_url,protocol,status,size
2068,www-c1.proxy.aol.com,1995-08-01 01:21:52-04:00,GET,/history/apollo/apollo.html,HTTP/1.0,200,3260
2069,www-c1.proxy.aol.com,1995-08-01 01:21:52-04:00,GET,/history/apollo/apollo.html,HTTP/1.0,200,3260
31384,www-d2.proxy.aol.com,1995-08-01 13:53:13-04:00,GET,/images/,HTTP/1.0,200,17688
31385,www-d2.proxy.aol.com,1995-08-01 13:53:13-04:00,GET,/images/,HTTP/1.0,200,17688
42865,www-a1.proxy.aol.com,1995-08-03 13:10:12-04:00,GET,/shuttle/missions/sts-69/images/images.html,HTTP/1.0,200,2443
42866,www-a1.proxy.aol.com,1995-08-03 13:10:12-04:00,GET,/shuttle/missions/sts-69/images/images.html,HTTP/1.0,200,2443
45074,128.159.146.40,1995-08-03 13:46:33-04:00,GET,/ksc.html,HTTP/1.0,200,7034
45075,128.159.146.40,1995-08-03 13:46:33-04:00,GET,/ksc.html,HTTP/1.0,200,7034
45206,192.215.71.55,1995-08-03 13:50:18-04:00,GET,/history/history.html,HTTP/1.0,200,1602
45207,192.215.71.55,1995-08-03 13:50:18-04:00,GET,/history/history.html,HTTP/1.0,200,1602


In [52]:
logs_data.drop_duplicates(inplace=True)

In [53]:
logs_data.duplicated().sum()

np.int64(0)

## FINAL DATA IS READY FOR ANALYSIS

### EXPORT TO CSV FILE

In [54]:
logs_data.to_csv("access_logs.csv", index=False)